In [1]:
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))



/kaggle/input/competitions/house-prices-advanced-regression-techniques/sample_submission.csv
/kaggle/input/competitions/house-prices-advanced-regression-techniques/data_description.txt
/kaggle/input/competitions/house-prices-advanced-regression-techniques/train.csv
/kaggle/input/competitions/house-prices-advanced-regression-techniques/test.csv


In [4]:
train = pd.read_csv("/kaggle/input/competitions/house-prices-advanced-regression-techniques/train.csv")
test = pd.read_csv("/kaggle/input/competitions/house-prices-advanced-regression-techniques/test.csv")

print(train.shape)
train.head()

(1460, 81)


,Id,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,...,PoolArea,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition,SalePrice
0,1,60,RL,65.0,8450,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,2,2008,WD,Normal,208500
1,2,20,RL,80.0,9600,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,5,2007,WD,Normal,181500
2,3,60,RL,68.0,11250,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,9,2008,WD,Normal,223500
3,4,70,RL,60.0,9550,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,2,2006,WD,Abnorml,140000
4,5,60,RL,84.0,14260,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,12,2008,WD,Normal,250000


In [5]:
train.isnull().sum().sort_values(ascending=False).head(20)

PoolQC          1453
MiscFeature     1406
Alley           1369
Fence           1179
MasVnrType       872
FireplaceQu      690
LotFrontage      259
GarageQual        81
GarageFinish      81
GarageType        81
GarageYrBlt       81
GarageCond        81
BsmtFinType2      38
BsmtExposure      38
BsmtCond          37
BsmtQual          37
BsmtFinType1      37
MasVnrArea         8
Electrical         1
Condition2         0
dtype: int64

In [7]:
cat_cols = train.select_dtypes(include=["object"]).columns

num_cols = train.select_dtypes(include=["number"]).columns

print("Categorical: ",len(cat_cols))
print("Numerical: ",len(num_cols))

Categorical:  43
Numerical:  38


In [9]:
for col in cat_cols:
    train[col] = train[col].fillna("None")
    test[col] = test[col].fillna("None")

for col in num_cols:
    train[col] = train[col].fillna(train[col].median())
    if col != "SalePrice":
        test[col] = test[col].fillna(test[col].median())


print(train.isnull().sum().sum())
print(test.isnull().sum().sum())

0
0


In [10]:
train = pd.get_dummies(train, columns=cat_cols,drop_first=True)
test = pd.get_dummies(test,columns=cat_cols,drop_first=True)

print(train.shape)
print(test.shape)

(1460, 262)
(1459, 249)


In [11]:
train,test = train.align(test,join="left",axis=1,fill_value=0)

print(train.shape)
print(test.shape)

(1460, 262)
(1459, 262)


In [13]:
x = train.drop(columns=["SalePrice"])
y = train["SalePrice"]

print(x.shape)
print(y.shape)

(1460, 261)
(1460,)


In [14]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error

x_train,x_val,y_train,y_val = train_test_split(x,y,test_size=0.2,random_state=42)

from xgboost import XGBRegressor

model = XGBRegressor(random_state = 42)
model.fit(x_train,y_train)

y_pred = model.predict(x_val)

rmse = np.sqrt(mean_squared_error(y_val,y_pred))
print(f"RMSE: {rmse}")

RMSE: 27823.735478903618


In [15]:
print(y.mean())
print(y.median())

180921.19589041095
163000.0


In [16]:
y_log = np.log1p(y)

x_train,x_val,y_train,y_val = train_test_split(x,y_log,test_size=0.2,random_state=42)

model = XGBRegressor(random_state=42)
model.fit(x_train,y_train)

y_pred = model.predict(x_val)

rmsle = np.sqrt(mean_squared_error(y_val,y_pred))
print(f"RMSLE: {rmsle}")

RMSLE: 0.14515386016803122


In [28]:
test_original = pd.read_csv("/kaggle/input/competitions/house-prices-advanced-regression-techniques/test.csv")
test = test.drop(columns=["SalePrice"],errors="ignore")

In [31]:
y_test_pred = model.predict(test)
y_test_pred = np.expm1(y_test_pred)

submission = pd.DataFrame({
    "Id": test_original["Id"],
    "SalePrice": y_test_pred
})

submission.to_csv("submission.csv",index=False)